# Generate strict-extractive 0.5B and 3B candidate banks for train, dev, and test

This notebook generates matched raw candidate banks from the current strict-extractive best-MRR SFT adapters for Qwen2.5-0.5B and Qwen2.5-3B. Both models use the same question order, full resources, strict-extractive prompt, random seed, and decoding settings, and run sequentially in separate subprocesses.

The full run generates 20 samples for 1,440 train, 160 dev, and 95 factoid test questions: 33,900 rows per model and 67,800 combined rows. It can take several hours, especially for 3B. Run the smoke configuration first; then change `RUN_MODE` to `"full"`, restart the kernel, and rerun every cell.

Dev and test candidates are diagnostics/final-evaluation artifacts only. The combined file records `split`; downstream DPO construction must select `split == "train"`.


## 1. Environment and paths

In [ ]:
from __future__ import annotations
import csv
import hashlib
import json
import os
import re
import subprocess
import sys
from collections import Counter, defaultdict
from pathlib import Path

os.environ.setdefault("HF_HUB_OFFLINE", "1")
os.environ.setdefault("TRANSFORMERS_OFFLINE", "1")
os.environ.setdefault("HF_DATASETS_OFFLINE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

def find_project_root(start: Path) -> Path:
    for parent in [start.resolve(), *start.resolve().parents]:
        for candidate in [parent, parent / "Task-Structured Counterfactual Preference Mining"]:
            if (candidate / "cse_dpo/generate_candidate_bank.py").is_file() and (candidate / "src").is_dir():
                return candidate.resolve()
    raise RuntimeError("Open this notebook inside the BioASQ_MRES workspace.")

PROJECT_ROOT = find_project_root(Path.cwd())
SOURCE_DIR = PROJECT_ROOT / "data/BioASQ_factoid_sft_prepared/single_answer_full_resources_qwen25_05b"
TEST_DIR = PROJECT_ROOT / "data/Task13BTest"
PROMPT_REGISTRY = PROJECT_ROOT / "prompts/factoid_single_answer_aligned.json"
PROMPT_ID = "factoid-single-answer-extractive-v1"
print("Python:", sys.executable, "| Project:", PROJECT_ROOT)


## 2. Freeze split inputs and verify the strict prompt


In [ ]:
RUN_MODE = "full"  # Use "smoke" for a short validation run.
MODELS_TO_GENERATE = ("05b", "3b")
SEED = 3407
FULL_SAMPLES_PER_QUESTION = 20
SMOKE_SAMPLES_PER_QUESTION = 2
SMOKE_QUESTIONS_PER_SPLIT = 2
MAX_SEQ_LENGTH = 9000
MAX_NEW_TOKENS = 64
TEMPERATURE = 0.7
TOP_P = 0.9
BATCH_SIZE = 1
LOAD_DTYPE = "float16"
SKIP_COMPLETE_BANKS = True

if RUN_MODE not in {"smoke", "full"}: raise ValueError("RUN_MODE must be smoke or full")
if set(MODELS_TO_GENERATE) != {"05b", "3b"}: raise ValueError("Generate both model sizes for a matched bank")
SAMPLES_PER_QUESTION = FULL_SAMPLES_PER_QUESTION if RUN_MODE == "full" else SMOKE_SAMPLES_PER_QUESTION
QUESTION_LIMIT = None if RUN_MODE == "full" else SMOKE_QUESTIONS_PER_SPLIT
label = "all_splits_s20_v2" if RUN_MODE == "full" else "all_splits_s2_smoke_v2"
OUTPUT_ROOT = PROJECT_ROOT / "Artifacts/cse_dpo/candidate_banks" / f"strict_extractive_qwen25_05b_3b_{label}_seed{SEED}"
COMBINED_BANK = OUTPUT_ROOT / "combined_candidate_bank.jsonl"
COMBINED_MANIFEST = OUTPUT_ROOT / "combined_manifest.json"
COMPARISON_CSV = OUTPUT_ROOT / "bank_comparison_by_question.csv"
MODEL_CONFIGS = {
 "05b": {"label":"strict_extractive_qwen25_05b_best_mrr", "ref":PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr", "sft":PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_05b_lora_dropout_005_strict_extractive/trainer_prepared/train_prepared.json"},
 "3b": {"label":"strict_extractive_qwen25_3b_best_mrr", "ref":PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/adapter_best_evidence_mrr", "sft":PROJECT_ROOT / "Artifacts/Factoid_SFT/models/evidence_grounded_per_supported_alias_qwen25_3b_lora_dropout_005_strict_extractive/trainer_prepared/train_prepared.json"},
}
for key,cfg in MODEL_CONFIGS.items():
 cfg.update(output_root=OUTPUT_ROOT/f"qwen25_{key}", bank_dir=OUTPUT_ROOT/f"qwen25_{key}")
 cfg["bank"] = cfg["bank_dir"] / "adapter-best-evidence-mrr/candidate_bank.jsonl"
 cfg["manifest"] = cfg["bank_dir"] / "adapter-best-evidence-mrr/manifest.json"
for key in MODELS_TO_GENERATE:
 assert (MODEL_CONFIGS[key]["ref"] / "adapter_config.json").exists()
 assert (MODEL_CONFIGS[key]["ref"] / "adapter_model.safetensors").exists()
 assert MODEL_CONFIGS[key]["sft"].exists()
assert (SOURCE_DIR/"train_prepared.json").exists() and (SOURCE_DIR/"eval_prepared.json").exists()
TEST_FILES = sorted(TEST_DIR.glob("*_golden.json"))
assert len(TEST_FILES) == 4
print("Mode:",RUN_MODE,"| samples/question:",SAMPLES_PER_QUESTION,"| output:",OUTPUT_ROOT)


## 3. Generate each split and model sequentially


In [ ]:
def sha256_file(path: Path) -> str:
    d=hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda:f.read(1024*1024),b""): d.update(block)
    return d.hexdigest()

def load_jsonl(path: Path):
    return [json.loads(x) for x in path.read_text(encoding="utf-8").splitlines() if x.strip()]

train_input = SOURCE_DIR / "train_prepared.json"
dev_input = SOURCE_DIR / "eval_prepared.json"
train_payload=json.loads(train_input.read_text(encoding="utf-8"))
dev_payload=json.loads(dev_input.read_text(encoding="utf-8"))
assert len(train_payload)==1440 and len(dev_payload)==160
def prepared_ids(payload): return [row["id"] for row in payload[:QUESTION_LIMIT] if isinstance(payload,list)]
def raw_factoid_ids(paths):
    out=[]
    for path in paths:
        payload=json.loads(path.read_text(encoding="utf-8"))
        out.extend(q["id"] for q in payload["questions"] if q.get("type")=="factoid")
    return out[:QUESTION_LIMIT] if QUESTION_LIMIT else out
SPLITS={"train":{"inputs":[train_input],"ids":prepared_ids(train_payload)}, "dev":{"inputs":[dev_input],"ids":prepared_ids(dev_payload)}, "test":{"inputs":TEST_FILES,"ids":raw_factoid_ids(TEST_FILES)}}
assert [len(SPLITS[s]["ids"]) for s in ("train","dev","test")] == ([2,2,2] if QUESTION_LIMIT else [1440,160,95])
assert not set(SPLITS["train"]["ids"]) & set(SPLITS["dev"]["ids"])
assert not set(SPLITS["train"]["ids"]) & set(SPLITS["test"]["ids"])
assert not set(SPLITS["dev"]["ids"]) & set(SPLITS["test"]["ids"])
registry=json.loads(PROMPT_REGISTRY.read_text(encoding="utf-8"))
strict_instruction=registry["prompts"][PROMPT_ID]["instructions"]["factoid"].strip()
for key,cfg in MODEL_CONFIGS.items():
    instructions={row["instruction"].strip() for row in json.loads(cfg["sft"].read_text(encoding="utf-8"))}
    assert instructions=={strict_instruction}, f"{key} SFT prompt differs from {PROMPT_ID}"
print({"train":len(SPLITS["train"]["ids"]),"dev":len(SPLITS["dev"]["ids"]),"test_factoid":len(SPLITS["test"]["ids"]),"disjoint":True,"prompt_id":PROMPT_ID})


## 4. Validate all banks and create one combined file


In [ ]:
def expected_rows(split): return len(SPLITS[split]["ids"])*SAMPLES_PER_QUESTION
def count_lines(path): return sum(bool(x.strip()) for x in path.open(encoding="utf-8"))
def bank_paths(split,key):
    cfg=MODEL_CONFIGS[key]
    bank_dir=cfg["output_root"]/split/"adapter-best-evidence-mrr"
    return bank_dir/"candidate_bank.jsonl", bank_dir/"manifest.json"
def completed(split,key):
    cfg=MODEL_CONFIGS[key]
    bank,manifest_path=bank_paths(split,key)
    if not bank.exists() or not manifest_path.exists(): return False
    try:
        m=json.loads(manifest_path.read_text(encoding="utf-8")); g=m["generation"]
        return all([count_lines(bank)==expected_rows(split),m.get("candidate_count")==expected_rows(split),m["dataset"]["question_count"]==len(SPLITS[split]["ids"]),[Path(x).resolve() for x in m["dataset"]["eval_input"]]==[x.resolve() for x in SPLITS[split]["inputs"]],Path(m["model"]["load_target"]).resolve()==cfg["ref"].resolve(),m["prompt"]["prompt_id"]==PROMPT_ID,g.get("samples_per_question_total")==SAMPLES_PER_QUESTION,g.get("seed")==SEED,g.get("max_seq_length")==MAX_SEQ_LENGTH,g.get("max_new_tokens")==MAX_NEW_TOKENS,g.get("temperature")==TEMPERATURE,g.get("top_p")==TOP_P,g.get("batch_size")==BATCH_SIZE])
    except (OSError,KeyError,TypeError,ValueError,json.JSONDecodeError): return False
def command(split,key):
    cfg=MODEL_CONFIGS[key]
    return [sys.executable,"-m","cse_dpo.generate_candidate_bank","--eval-input",*[str(x) for x in SPLITS[split]["inputs"]],"--model-ref",str(cfg["ref"]),"--output-dir",str(cfg["output_root"]/split),"--dataset-name",f"bioasq-factoid-{split}","--question-types","factoid","--prompt-file",str(PROMPT_REGISTRY),"--prompt",PROMPT_ID,"--chat-template","qwen-2.5","--prompt-format","chat","--samples-per-question-total",str(SAMPLES_PER_QUESTION),"--max-resources","0","--max-resource-chars","0","--resource-selection","first","--resource-granularity","document","--max-seq-length",str(MAX_SEQ_LENGTH),"--max-new-tokens",str(MAX_NEW_TOKENS),"--temperature",str(TEMPERATURE),"--top-p",str(TOP_P),"--batch-size",str(BATCH_SIZE),"--dtype",LOAD_DTYPE,"--device-map","auto","--seed",str(SEED),"--factoid-parser-mode","agnostic","--local-files-only"] + (["--limit",str(QUESTION_LIMIT)] if QUESTION_LIMIT else [])
def generate(split,key):
    if SKIP_COMPLETE_BANKS and completed(split,key): print(f"[{split}/{key}] complete; skipping"); return
    print(f"[{split}/{key}] generating {expected_rows(split)} rows")
    subprocess.run(command(split,key),cwd=PROJECT_ROOT,env=os.environ.copy(),check=True)
    if not completed(split,key): raise RuntimeError(f"Completion validation failed: {split}/{key}")
gpu_probe=subprocess.run([sys.executable,"-c","import torch; print(int(torch.cuda.is_available()))"],cwd=PROJECT_ROOT,capture_output=True,text=True,check=True)
if gpu_probe.stdout.strip().splitlines()[-1]!="1": raise RuntimeError("CUDA is unavailable; refusing a long CPU generation run.")
for split in ("train","dev","test"):
    for key in MODELS_TO_GENERATE: generate(split,key)
print("All train/dev/test banks completed.")


## 5. Compare 0.5B and 3B within each split


In [ ]:
def validate(split,key):
    cfg=MODEL_CONFIGS[key]; assert completed(split,key)
    bank,manifest_path=bank_paths(split,key)
    m=json.loads(manifest_path.read_text(encoding="utf-8")); seen=defaultdict(list); statuses=Counter(); checkpoints=set()
    for line in bank.open(encoding="utf-8"):
        if not line.strip(): continue
        row=json.loads(line); seen[row["question_id"]].append(row["sample_id"]); statuses[row.get("parser_status","missing")]+=1; checkpoints.add(str(Path(row["generator_checkpoint"]).resolve()))
    assert list(seen)==SPLITS[split]["ids"]
    assert all(v==list(range(SAMPLES_PER_QUESTION)) for v in seen.values())
    assert checkpoints=={str(cfg["ref"].resolve())}
    assert not m["generation"]["prompt_truncation"].get("prompt_truncated")
    return {"split":split,"model":key,"questions":len(seen),"candidates":sum(map(len,seen.values())),"parser_status":dict(statuses),"bank_sha256":sha256_file(bank)}
validation=[validate(split,key) for split in ("train","dev","test") for key in MODELS_TO_GENERATE]
print(json.dumps(validation,indent=2))
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
def combined_rows():
    for split in ("train","dev","test"):
        for key in MODELS_TO_GENERATE:
            bank=MODEL_CONFIGS[key]["output_root"]/split/"adapter-best-evidence-mrr/candidate_bank.jsonl"
            for line in bank.open(encoding="utf-8"):
                if line.strip():
                    row=json.loads(line); source=row["response_id"]; yield {**row,"split":split,"source_response_id":source,"response_id":f"{split}::{key}::{source}","generator_model_size":key,"generator_model_label":MODEL_CONFIGS[key]["label"],"source_candidate_bank":str(bank)}
tmp=COMBINED_BANK.with_suffix(".tmp")
with tmp.open("w",encoding="utf-8") as f:
    combined_count=0
    for row in combined_rows(): f.write(json.dumps(row,ensure_ascii=False,sort_keys=True)+"\n"); combined_count+=1
tmp.replace(COMBINED_BANK)
expected=sum(expected_rows(s) for s in ("train","dev","test"))*len(MODELS_TO_GENERATE)
assert combined_count==expected
manifest={"status":"complete","run_mode":RUN_MODE,"splits":["train","dev","test"],"question_counts":{s:len(SPLITS[s]["ids"]) for s in SPLITS},"models":list(MODELS_TO_GENERATE),"samples_per_question_per_model":SAMPLES_PER_QUESTION,"combined_candidate_count":combined_count,"sampling":{"seed":SEED,"temperature":TEMPERATURE,"top_p":TOP_P,"batch_size":BATCH_SIZE,"max_new_tokens":MAX_NEW_TOKENS},"context":{"full_mixed_resources":True,"max_resources":0,"max_resource_chars":0,"max_seq_length":MAX_SEQ_LENGTH,"prompt_truncation_forbidden":True},"prompt":{"id":PROMPT_ID,"registry":str(PROMPT_REGISTRY),"registry_sha256":sha256_file(PROMPT_REGISTRY)},"model_sources":{k:{"checkpoint":str(MODEL_CONFIGS[k]["ref"]),"adapter_config_sha256":sha256_file(MODEL_CONFIGS[k]["ref"]/"adapter_config.json"),"adapter_weights_sha256":sha256_file(MODEL_CONFIGS[k]["ref"]/"adapter_model.safetensors")} for k in MODELS_TO_GENERATE},"combined_candidate_bank":str(COMBINED_BANK),"combined_candidate_bank_sha256":sha256_file(COMBINED_BANK),"dpo_rule":"Only split=train rows may enter preference-data construction."}
COMBINED_MANIFEST.write_text(json.dumps(manifest,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print("Combined:",COMBINED_BANK)


## 6. Interpretation and next use

The combined file contains raw generations only. Use `split == "train"` when constructing HEA annotations or DPO pairs. Keep `dev` for checkpoint selection and diagnostics, and keep `test` untouched for final reporting.


In [ ]:
def norm(text): return re.sub(r"\s+"," ",re.sub(r"[^a-z0-9]+"," ",re.sub(r"\[(?:BE|EE)\]"," ",text or "",flags=re.I).casefold())).strip()
comparison=[]
for split in ("train","dev","test"):
    per={key:defaultdict(list) for key in MODELS_TO_GENERATE}
    for key in MODELS_TO_GENERATE:
        bank=MODEL_CONFIGS[key]["output_root"]/split/"adapter-best-evidence-mrr/candidate_bank.jsonl"
        for line in bank.open(encoding="utf-8"):
            if line.strip():
                row=json.loads(line); per[key][row["question_id"]].append(norm(row.get("raw_output","")))
    for qid in SPLITS[split]["ids"]:
        left={x for x in per["05b"][qid] if x}; right={x for x in per["3b"][qid] if x}; union=left|right
        comparison.append({"split":split,"question_id":qid,"unique_05b":len(left),"unique_3b":len(right),"shared_unique_outputs":len(left&right),"jaccard":len(left&right)/len(union) if union else 1.0})
with COMPARISON_CSV.open("w",encoding="utf-8",newline="") as f:
    writer=csv.DictWriter(f,fieldnames=list(comparison[0])); writer.writeheader(); writer.writerows(comparison)
print(json.dumps({s:{"questions":sum(x["split"]==s for x in comparison),"mean_jaccard":sum(x["jaccard"] for x in comparison if x["split"]==s)/sum(x["split"]==s for x in comparison)} for s in ("train","dev","test")},indent=2))


## 6. Combine both raw banks without losing provenance

The combined file retains every sampled row. Exact duplicate answers across models are intentionally preserved because their generating model and sample frequency are part of the evidence. `response_id` is namespaced by model; the original ID is kept as `source_response_id`.

In [ ]:
# Superseded: cell 10 already combines the split-specific banks.
print("Combined bank was created by the all-splits validation cell.")

## 7. Compare candidate diversity and cross-model overlap

In [ ]:
# Superseded: cell 12 performs the all-splits comparison.
print("Cross-model comparison was created by the all-splits comparison cell.")
"""
def normalized_answer(text: str) -> str:
    text = re.sub(r"\[(?:BE|EE)\]", " ", text or "", flags=re.IGNORECASE)
    text = re.sub(r"[^a-z0-9]+", " ", text.casefold())
    return re.sub(r"\s+", " ", text).strip()

outputs = {key: defaultdict(list) for key in MODELS_TO_GENERATE}
parser_counts = {key: Counter() for key in MODELS_TO_GENERATE}
for key in MODELS_TO_GENERATE:
    with MODEL_CONFIGS[key]["bank"].open(encoding="utf-8") as handle:
        for line in handle:
            if not line.strip():
                continue
            row = json.loads(line)
            outputs[key][row["question_id"]].append(normalized_answer(row.get("raw_output", "")))
            parser_counts[key][row.get("parser_status", "missing")] += 1

comparison_rows = []
if set(MODELS_TO_GENERATE) == {"05b", "3b"}:
    for qid in selected_ids:
        left = {value for value in outputs["05b"][qid] if value}
        right = {value for value in outputs["3b"][qid] if value}
        union = left | right
        comparison_rows.append({
            "question_id": qid,
            "unique_05b": len(left),
            "unique_3b": len(right),
            "shared_unique_outputs": len(left & right),
            "union_unique_outputs": len(union),
            "jaccard": len(left & right) / len(union) if union else 1.0,
        })
    with COMPARISON_CSV.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(comparison_rows[0]))
        writer.writeheader()
        writer.writerows(comparison_rows)
    aggregate = {
        "questions": len(comparison_rows),
        "mean_unique_05b": sum(row["unique_05b"] for row in comparison_rows) / len(comparison_rows),
        "mean_unique_3b": sum(row["unique_3b"] for row in comparison_rows) / len(comparison_rows),
        "mean_shared_unique_outputs": sum(row["shared_unique_outputs"] for row in comparison_rows) / len(comparison_rows),
        "mean_jaccard": sum(row["jaccard"] for row in comparison_rows) / len(comparison_rows),
        "questions_with_no_cross_model_overlap": sum(row["shared_unique_outputs"] == 0 for row in comparison_rows),
        "parser_status": {key: dict(value) for key, value in parser_counts.items()},
        "per_question_csv": str(COMPARISON_CSV),
    }
    print(json.dumps(aggregate, indent=2))
else:
    print("Cross-model comparison requires both 05b and 3b; parser counts:", {key: dict(value) for key, value in parser_counts.items()})
"""


## 8. What to do next

For the full run, use `combined_candidate_bank.jsonl` as the raw input to a new HEA annotation build. Preserve `generator_model_size` during deduplication so we can measure whether an error came from 0.5B, 3B, or both.

Recommended construction order:

1. Insert each question's recorded BioASQ gold aliases as the only benchmark-gold C3 members.
2. Deduplicate generated answer surfaces within each question while retaining model/sample provenance and frequency.
3. Classify generated candidates for semantic correctness, completeness, boundary relation, and evidence support.
4. Exclude unresolved or ambiguous candidates rather than treating every non-gold surface as a negative.
5. Freeze one reviewed dataset and use it unchanged for matched standard-DPO and HEA-DPO runs.

Do not combine development annotations or development generations into the training bank.